In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Clone your collaboration repository
# Replace with your actual GitHub URL
#!git clone https://github.com/nmorok/Teleconnections-ViT.git
!git clone -b final_boss --single-branch https://github.com/nmorok/Teleconnections-ViT.git

# 2. Enter the repository directory
import os
os.chdir('Teleconnections-ViT')
# 3. Add the current directory to sys.path so 'import model' works
import sys
sys.path.append(os.getcwd())

# 4. Verify the files are present
print("Files in current directory:", os.listdir())

Cloning into 'Teleconnections-ViT'...
remote: Enumerating objects: 1181, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 1181 (delta 9), reused 13 (delta 2), pack-reused 1144 (from 1)
Receiving objects: 100% (1181/1181), 1.77 GiB | 17.45 MiB/s, done.
Resolving deltas: 100% (468/468), done.
Updating files: 100% (516/516), done.
Files in current directory: ['data', 'models', '.gitignore', 'LICENSE', 'Pipeline.md', 'Methods.md', 'requirements.txt', '.git', 'notebook', 'change_log.md', 'README.md', 'grid_mask_verification.png', 'CrabTransformer_Methods.md']


In [ ]:
from google.colab import drive
import torch
import math


# 2. Define your data path (Update this to your actual Drive folder)
# Usually formatted as: /content/drive/MyDrive/Folder_Name
DATA_PATH = '/content/drive/MyDrive/Teleconnection_ViT/data'
REPO_DIR = '/content/Teleconnections-ViT'

# 3. Quick verification check
if os.path.exists(DATA_PATH):
    print(f"✓ Data folder found at: {DATA_PATH}")
    print("Files available:", os.listdir(DATA_PATH))
else:
    print(f"✗ ERROR: Could not find folder at {DATA_PATH}. Check your Drive path.")

# 4. Hardware Check
# 1. Check if CUDA (GPU support) is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 2. Get the name of the GPU assigned by Colab Pro
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Using CPU. Check your Colab Runtime settings.")

✓ Data folder found at: /content/drive/MyDrive/Teleconnection_ViT/data
Files available: ['easy', 'medium', 'hard']
Is CUDA available? True
GPU Name: NVIDIA A100-SXM4-80GB


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
import math

from torch.utils.data import DataLoader
from models.model import CrabTransformer
from models.losses import TweedieLoss, MSELoss_cm
from data.data_helper import CrabDataset, get_dataloaders

The below is looking to make sure the transformations and the dataloaders aren't impacting the data.

I'll eventually look at the data post transformer model to make sure it looks similar

In [ ]:
"""
validate_configs.py
====================
Checks that every channel-config × prediction-mode combination in your run
matrix loads data correctly, builds the model, and completes a forward pass —
all on CPU, in seconds, with no training required.

What it does
------------
For each combination it:
  1. Calls get_dataloaders() and pulls ONE batch from the training loader.
  2. Verifies tensor shapes, channel count, and mask length.
  3. Instantiates CrabTransformer with the dataset's channel metadata.
  4. Runs one forward pass and checks output shape.
  5. Runs one forward pass with return_attention=True and checks attention shape.
  6. Prints a PASS / FAIL line with key numbers.

After testing all configs it:
  • Prints a summary table (all results at a glance).
  • Saves a figure for each tested data type (dummy / real) showing:
      - One row per channel config.
      - Columns = each channel in the input tensor, shown as a 50×50 heatmap
        (first sample, first channel group, mean across lookbacks for history
        channels to keep the grid manageable).

Output files
------------
  validation_results.csv   — pass/fail + shapes for every config
  validation_dummy.png     — channel grid for dummy data
  validation_real.png      — channel grid for real data  (if real data exists)

Usage
-----
  python validate_configs.py            # tests dummy only (safe if no real data)
  python validate_configs.py --real     # also test real data configs

All paths are relative to REPO_DIR; adjust the two constants at the top.
"""

import sys
import os
import argparse
import traceback
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

# ---- adjust to your layout ----------------------------------------
REPO_DIR   = '/content/Teleconnections-ViT'
OUTPUT_DIR = '/content/drive/MyDrive/Teleconnection_ViT/validation'
# -------------------------------------------------------------------

sys.path.insert(0, REPO_DIR)
from data.data_helper  import get_dataloaders, get_channel_info
from models.model      import CrabTransformer


# ============================================================
#  CONFIG TABLES  (mirror train.py exactly)
# ============================================================

MODEL_CONFIGS = {
    'normal': {'embed_dim': 128, 'num_heads': 8, 'num_layers': 6,
               'd_ff': 512, 'dropout': 0.0},
    'small':  {'embed_dim': 128, 'num_heads': 4, 'num_layers': 3,
               'd_ff': 512, 'dropout': 0.0},
}

CHANNEL_CONFIGS = {
    'all':           {'use_spawners': True,  'use_recruits': True,  'use_temp': True},
    'sp_rec':        {'use_spawners': True,  'use_recruits': True,  'use_temp': False},
    'sp_temp':       {'use_spawners': True,  'use_recruits': False, 'use_temp': True},
    'rec_temp':      {'use_spawners': False, 'use_recruits': True,  'use_temp': True},
    'spawners_only': {'use_spawners': True,  'use_recruits': False, 'use_temp': False},
    'recruits_only': {'use_spawners': False, 'use_recruits': True,  'use_temp': False},
    'temp_only':     {'use_spawners': False, 'use_recruits': False, 'use_temp': True},
}

PREDICTION_MODES = {
    'normal':         {'incl_curr': True,  'lag': 0},
    'one_year_ahead': {'incl_curr': False, 'lag': 0},
    'lag5':           {'incl_curr': True,  'lag': 5},
}

# Human-readable channel names in tensor order.
# include_current mirrors data_helper.get_channel_info — when False,
# current-year spawner and temp channels are absent from the tensor.
def channel_names_for(use_spawners, use_recruits, use_temp,
                      include_current: bool = True):
    names = []
    if use_spawners:
        if include_current:
            names += ['Sp(t)']
        names += ['Sp(t-1)', 'Sp(t-2)', 'Sp(t-3)', 'Sp(t-4)', 'Sp(t-5)']
    if use_recruits:
        names += ['Rc(t-1)', 'Rc(t-2)', 'Rc(t-3)', 'Rc(t-4)', 'Rc(t-5)']
    if use_temp:
        if include_current:
            names += ['Tmp(t)']
        names += ['Tmp(t-1)', 'Tmp(t-2)', 'Tmp(t-3)', 'Tmp(t-4)', 'Tmp(t-5)']
    return names


# ============================================================
#  YEAR SPLITS
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    return (18, 9, 3)


# ============================================================
#  RESULT DATACLASS
# ============================================================

@dataclass
class ConfigResult:
    data_type:    str
    level:        str
    channel_cfg:  str
    pred_mode:    str
    model_size:   str
    passed:       bool
    error:        str          = ''
    in_channels:  int          = 0
    input_shape:  str          = ''
    output_shape: str          = ''
    mask_shape:   str          = ''
    attn_shape:   str          = ''
    mask_indices: str          = ''
    n_valid_yr:   int          = 0       # non-zero valid_year flags in batch
    input_mean:   float        = 0.0
    input_std:    float        = 0.0
    target_mean:  float        = 0.0
    sample_tensor: Optional[np.ndarray] = field(default=None, repr=False)
    channel_names: list        = field(default_factory=list, repr=False)


# ============================================================
#  SINGLE CONFIG TEST
# ============================================================

def test_config(data_type, level, channel_cfg, pred_mode,
                model_size='normal') -> ConfigResult:
    """
    Load one batch, run one forward pass, return a ConfigResult.
    Uses batch_size=2 and only reads the first batch — very fast on CPU.
    """
    result = ConfigResult(
        data_type=data_type, level=level, channel_cfg=channel_cfg,
        pred_mode=pred_mode, model_size=model_size, passed=False,
    )

    ccfg = CHANNEL_CONFIGS[channel_cfg]
    pcfg = PREDICTION_MODES[pred_mode]
    t_years, v_years, te_years = get_year_splits(data_type, pcfg['lag'])

    try:
        # -- Data --
        train_loader, _, _ = get_dataloaders(
            batch_size=2, memory_years=5,
            train_years=t_years, val_years=v_years, test_years=te_years,
            level=level, data_type=data_type,
            include_current_spawner=pcfg['incl_curr'],
            lag=pcfg['lag'],
            use_temp=ccfg['use_temp'],
            use_spawners=ccfg['use_spawners'],
            use_recruits=ccfg['use_recruits'],
        )

        ds = train_loader.dataset
        result.in_channels  = ds.in_channels
        result.mask_indices = str(ds.channel_mask_indices)
        result.channel_names = channel_names_for(
            ccfg['use_spawners'], ccfg['use_recruits'],
            ds.use_temp,           # actual temp (False for dummy)
            include_current=pcfg['incl_curr'],
        )

        # Pull ONE batch
        batch = next(iter(train_loader))
        inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch

        result.input_shape  = str(list(inputs.shape))
        result.mask_shape   = str(list(temporal_mask.shape))
        result.target_mean  = float(targets.mean())
        result.input_mean   = float(inputs.mean())
        result.input_std    = float(inputs.std())
        result.n_valid_yr   = int((valid_year > 0).sum())

        # Shape assertions
        B = inputs.shape[0]
        assert inputs.shape == (B, ds.in_channels, 50, 50), \
            f"Input shape mismatch: {inputs.shape}"
        assert targets.shape == (B, 1, 50, 50), \
            f"Target shape mismatch: {targets.shape}"
        assert temporal_mask.shape == (B, 6), \
            f"Temporal mask shape mismatch: {temporal_mask.shape}"
        assert len(ds.channel_mask_indices) == ds.in_channels, \
            f"channel_mask_indices length {len(ds.channel_mask_indices)} ≠ in_channels {ds.in_channels}"

        # -- Model --
        mcfg  = MODEL_CONFIGS[model_size]
        model = CrabTransformer(
            grid_size=50, patch_size=5,
            in_channels=ds.in_channels,
            embed_dim=mcfg['embed_dim'],
            num_heads=mcfg['num_heads'],
            num_layers=mcfg['num_layers'],
            d_ff=mcfg['d_ff'],
            dropout=mcfg['dropout'],
            channel_mask_indices=ds.channel_mask_indices,
        )
        model.eval()

        with torch.no_grad():
            # Normal forward
            output = model(inputs, year_idx, temporal_mask,
                           spatial_mask=spatial_mask)
            result.output_shape = str(list(output.shape))
            assert output.shape == (B, 1, 50, 50), \
                f"Output shape mismatch: {output.shape}"

            # With attention
            output_a, attn_maps = model(inputs, year_idx, temporal_mask,
                                        spatial_mask=spatial_mask,
                                        return_attention=True)
            assert len(attn_maps) == mcfg['num_layers'], \
                f"Expected {mcfg['num_layers']} attention maps, got {len(attn_maps)}"
            expected_heads = mcfg['num_heads']
            assert attn_maps[0].shape[1] == expected_heads, \
                f"Attention head count mismatch"
            result.attn_shape = str(list(attn_maps[0].shape))

            # Output is non-negative (softplus activated)
            assert float(output.min()) >= 0.0, "Output contains negative values"

            # Masked cells (outside EBS region) should be exactly zero.
            # Guard against dummy data where spatial_mask is all-ones
            # — indexing an empty selection with .max() raises an error.
            outside = (spatial_mask == 0).unsqueeze(1)
            if outside.any():
                assert float(output[outside].max()) == 0.0, \
                    "Non-zero values found outside spatial mask"

        # Store the raw input tensor for visualisation (first sample)
        result.sample_tensor = inputs[0].numpy()   # [C, 50, 50]
        result.passed        = True

    except Exception as e:
        result.error = f"{type(e).__name__}: {e}"
        # Print traceback for debugging
        traceback.print_exc()

    return result


# ============================================================
#  VISUALISATION  — channel grid for a set of results
# ============================================================

CMAP_BY_GROUP = {
    'Sp': 'YlOrRd',
    'Rc': 'Blues',
    'Tm': 'RdBu_r',
}

def group_color(name):
    if name.startswith('Sp'):   return CMAP_BY_GROUP['Sp']
    if name.startswith('Rc'):   return CMAP_BY_GROUP['Rc']
    return CMAP_BY_GROUP['Tm']


def make_channel_figure(results: list, title: str, save_path: str):
    """
    Grid figure: one row per channel config, one column per channel.
    The widest config (most channels) determines the number of columns.
    Narrower configs leave trailing cells blank.
    Shows the first sample's spatial grid for each channel.
    """
    # Only passed results
    passed = [r for r in results if r.passed and r.sample_tensor is not None]
    if not passed:
        print("  No passed results with tensors — skipping figure.")
        return

    max_ch   = max(r.in_channels for r in passed)
    n_rows   = len(passed)
    n_cols   = max_ch + 1    # +1 for the row-label column
    fig_w    = min(2.2 * n_cols + 1.5, 40)
    fig_h    = min(2.4 * n_rows + 1.0, 60)

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=100)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(
        n_rows, n_cols,
        figure=fig,
        hspace=0.55, wspace=0.08,
        left=0.01, right=0.99, top=0.97, bottom=0.03,
    )

    for row_idx, r in enumerate(passed):
        ch_names = r.channel_names
        tensor   = r.sample_tensor          # [C, 50, 50]

        # Row label (left-most column, spanning the full height)
        ax_lbl = fig.add_subplot(gs[row_idx, 0])
        mode_short = {'normal': 'norm', 'one_year_ahead': '1yr↑', 'lag5': 'lag5'}
        lbl = (f"{r.channel_cfg}\n"
               f"{mode_short.get(r.pred_mode, r.pred_mode)}\n"
               f"{r.model_size}\n"
               f"[{r.in_channels}ch]")
        ax_lbl.text(0.5, 0.5, lbl, ha='center', va='center',
                    fontsize=7, fontweight='bold', transform=ax_lbl.transAxes,
                    multialignment='center')
        ax_lbl.axis('off')

        # One subplot per channel
        for c_idx in range(max_ch):
            ax = fig.add_subplot(gs[row_idx, c_idx + 1])

            if c_idx < r.in_channels:
                img  = tensor[c_idx]
                name = ch_names[c_idx] if c_idx < len(ch_names) else f'ch{c_idx}'
                cmap = group_color(name)
                vmin, vmax = float(img.min()), float(img.max())
                if vmax - vmin < 1e-6:
                    vmin, vmax = 0.0, 1.0
                ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax,
                          interpolation='nearest', aspect='equal')
                ax.set_title(name, fontsize=6, pad=2)

                # Show temporal mask slot
                midx = r.sample_tensor.shape[0]   # just a fallback
                if hasattr(r, 'mask_indices') and r.mask_indices:
                    try:
                        mlist = eval(r.mask_indices)
                        if c_idx < len(mlist):
                            midx = mlist[c_idx]
                            ax.set_xlabel(f'm[{midx}]', fontsize=5, labelpad=1)
                    except Exception:
                        pass

                # Frame colour by channel group
                edge = {'Sp': '#d62728', 'Rc': '#1f77b4', 'Tm': '#2ca02c'}
                grp  = name[:2] if len(name) >= 2 else '??'
                for sp in ax.spines.values():
                    sp.set_edgecolor(edge.get(grp, 'grey'))
                    sp.set_linewidth(1.5)
            else:
                # Empty cell for configs with fewer channels
                ax.set_facecolor('#f0f0f0')
                for sp in ax.spines.values():
                    sp.set_visible(False)

            ax.set_xticks([])
            ax.set_yticks([])

    # Legend
    from matplotlib.patches import Patch
    legend_els = [
        Patch(facecolor='#d62728', label='Spawner channels'),
        Patch(facecolor='#1f77b4', label='Recruit channels'),
        Patch(facecolor='#2ca02c', label='Temperature channels'),
        Patch(facecolor='#f0f0f0', label='Unused slot'),
    ]
    fig.legend(handles=legend_els, loc='lower center',
               ncol=4, fontsize=8, frameon=True,
               bbox_to_anchor=(0.5, -0.01))

    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"  Figure saved → {save_path}")


# ============================================================
#  SUMMARY TABLE
# ============================================================

def print_summary_table(results: list):
    rows = []
    for r in results:
        status = '✅ PASS' if r.passed else f'❌ FAIL'
        rows.append({
            'status':      status,
            'data':        r.data_type[:4],
            'level':       r.level[:5],
            'channel_cfg': r.channel_cfg,
            'pred_mode':   r.pred_mode,
            'model':       r.model_size,
            'in_ch':       r.in_channels,
            'input_shape': r.input_shape,
            'output':      r.output_shape,
            'attn':        r.attn_shape,
            'n_valid':     r.n_valid_yr,
            'inp_mean':    f'{r.input_mean:.3f}',
            'tgt_mean':    f'{r.target_mean:.3f}',
            'error':       r.error[:60] if r.error else '',
        })
    df = pd.DataFrame(rows)

    # Wide print
    pd.set_option('display.max_rows', 300)
    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 220)
    pd.set_option('display.max_colwidth', 65)

    n_pass = sum(r.passed for r in results)
    n_fail = len(results) - n_pass

    print(f"\n{'='*120}")
    print(f"  VALIDATION SUMMARY  —  {n_pass} passed, {n_fail} failed "
          f"out of {len(results)} tested")
    print(f"{'='*120}")
    print(df.to_string(index=False))

    if n_fail > 0:
        print(f"\n{'─'*120}")
        print("FAILURES:")
        for r in results:
            if not r.passed:
                print(f"  {r.data_type}/{r.level}/{r.channel_cfg}/"
                      f"{r.pred_mode}/{r.model_size}  →  {r.error}")
    print(f"{'='*120}\n")

    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    return df


# ============================================================
#  MAIN
# ============================================================

def main(test_real: bool = False):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_results = []

    # ---- Build test matrix ----
    # One model size (normal) is enough to validate data pipeline.
    # Both model sizes are tested for the architecture checks.
    # We use a single dummy level (easy) for speed.

    test_configs = []

    # Dummy: all no-temp channel configs × all pred modes.
    # one_year_ahead is skipped only for recruits_only (no current-year channel).
    for ch_cfg, ccfg in CHANNEL_CONFIGS.items():
        if ccfg['use_temp']:
            continue     # no temp files for dummy
        for pred_mode, pcfg in PREDICTION_MODES.items():
            if pred_mode == 'lag5':
                continue   # no lag5 splits for dummy
            if pred_mode == 'one_year_ahead' and \
                    not (ccfg['use_spawners'] or ccfg['use_temp']):
                continue   # no-op: recruits_only has no current-year channel without spawners
            # Test with both model sizes
            for model_size in MODEL_CONFIGS:
                test_configs.append(dict(
                    data_type='dummy', level='easy',
                    channel_cfg=ch_cfg, pred_mode=pred_mode,
                    model_size=model_size,
                ))

    # Real: all 7 channel configs × all applicable modes × both model sizes
    if test_real:
        for ch_cfg, ccfg in CHANNEL_CONFIGS.items():
            for pred_mode, pcfg in PREDICTION_MODES.items():
                if pred_mode == 'one_year_ahead' and \
                        not (ccfg['use_spawners'] or ccfg['use_temp']):
                    continue   # no-op: recruits_only has no current-year channel
                for model_size in MODEL_CONFIGS:
                    test_configs.append(dict(
                        data_type='real', level='real',
                        channel_cfg=ch_cfg, pred_mode=pred_mode,
                        model_size=model_size,
                    ))

    n = len(test_configs)
    print(f"\n{'='*72}")
    print(f"  CrabTransformer  —  Config Validator")
    print(f"  {n} configurations to test  |  CPU-only, no training")
    print(f"  Output: {OUTPUT_DIR}")
    print(f"{'='*72}\n")

    for i, cfg in enumerate(test_configs, 1):
        label = (f"{cfg['data_type']}/{cfg['level']}/"
                 f"{cfg['channel_cfg']}/{cfg['pred_mode']}/{cfg['model_size']}")
        print(f"[{i:3d}/{n}]  Testing: {label} … ", end='', flush=True)
        result = test_config(**cfg)
        status = '✅' if result.passed else f'❌  {result.error[:80]}'
        print(status)
        all_results.append(result)

    # ---- Summary table ----
    df_summary = print_summary_table(all_results)

    # Save CSV
    csv_path = os.path.join(OUTPUT_DIR, 'validation_results.csv')
    # Drop the large tensor field before saving
    df_summary.to_csv(csv_path, index=False)
    print(f"Results saved → {csv_path}\n")

    # ---- Channel grid figures ----
    print("Generating channel grid figures …")

    # Dummy figure: one row per channel_cfg × pred_mode combo,
    # using 'normal' model size (both sizes test identically for data)
    dummy_results = [r for r in all_results
                     if r.data_type == 'dummy' and r.model_size == 'normal']
    if dummy_results:
        make_channel_figure(
            dummy_results,
            title='Dummy data — channel layout per config  '
                  '(red=spawner  blue=recruit  green=temp)',
            save_path=os.path.join(OUTPUT_DIR, 'validation_dummy.png'),
        )

    real_results = [r for r in all_results
                    if r.data_type == 'real' and r.model_size == 'normal']
    if real_results:
        make_channel_figure(
            real_results,
            title='Real data — channel layout per config  '
                  '(red=spawner  blue=recruit  green=temp)',
            save_path=os.path.join(OUTPUT_DIR, 'validation_real.png'),
        )

    # ---- Quick stats per channel config (data perspective) ----
    print("\n=== PER-CONFIG CHANNEL STATS (first batch, first sample) ===")
    seen = set()
    for r in all_results:
        key = (r.data_type, r.level, r.channel_cfg, r.pred_mode)
        if not r.passed or key in seen or r.sample_tensor is None:
            continue
        seen.add(key)
        tensor = r.sample_tensor   # [C, 50, 50]
        print(f"\n  {r.data_type}/{r.channel_cfg}/{r.pred_mode}  "
              f"[{r.in_channels} channels]")
        print(f"  {'Channel':<12}  {'mask_idx':>8}  "
              f"{'min':>7}  {'mean':>7}  {'max':>7}  "
              f"{'zeros%':>7}  {'nonzero_mean':>12}")
        mask_idx = eval(r.mask_indices)
        for c_idx in range(tensor.shape[0]):
            img = tensor[c_idx]
            mn, mu, mx = img.min(), img.mean(), img.max()
            z_pct = (img == 0).mean() * 100
            nz_mu = img[img != 0].mean() if (img != 0).any() else 0.0
            name  = r.channel_names[c_idx] if c_idx < len(r.channel_names) else f'ch{c_idx}'
            midx  = mask_idx[c_idx] if c_idx < len(mask_idx) else '?'
            print(f"  {name:<12}  {midx:>8}  "
                  f"{mn:>7.3f}  {mu:>7.3f}  {mx:>7.3f}  "
                  f"{z_pct:>6.1f}%  {nz_mu:>12.3f}")

    n_pass = sum(r.passed for r in all_results)
    n_fail = len(all_results) - n_pass
    print(f"\n{'='*72}")
    if n_fail == 0:
        print(f"  ✅  All {n_pass} configurations passed.")
    else:
        print(f"  ⚠   {n_pass} passed, {n_fail} FAILED.  "
              f"Check the table above for error details.")
    print(f"{'='*72}\n")


# ============================================================
#  ENTRY POINT
# ============================================================

if __name__ == '__main__':
    main(test_real=True)

In [ ]:
!pip install sympy==1.13.3 --quiet

In [ ]:
"""
train.py  —  Master training script for all CrabTransformer configurations.

Directory layout
----------------
model_outputs/
  {model_size}/               normal | small
    {level}/                  easy | medium | hard | real
      {channel_cfg}/          all | sp_rec | sp_temp | rec_temp |
                              spawners_only | recruits_only | temp_only
        {pred_mode}/          normal | one_year_ahead | lag5
          {criterion}/        MSE | Tweedie
            best_model.pt
            training_history.json
            training_curves.png

Channel configurations (7 total)
----------------------------------
  all           spawners (6ch) + recruits (5ch) + temp (6ch)  = 17ch
  sp_rec        spawners + recruits                            = 11ch
  sp_temp       spawners + temp                               = 12ch
  rec_temp      recruits + temp                               = 11ch
  spawners_only spawners only                                 =  6ch
  recruits_only recruits only                                 =  5ch
  temp_only     temp only                                     =  6ch

Run matrix rules
----------------
* Dummy (easy/medium/hard): no temp files → skip configs that need temp.
* one_year_ahead: only valid when spawners are in the config (otherwise the
  mode has no effect — channel 0 doesn't exist to zero out).
* lag5: only valid for real data (separate split directories exist for lag=5).
* SKIP_IF_EXISTS=True  lets you resume an interrupted run safely.
"""

import os
import sys
import json
import traceback

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ---- adjust to your Colab / Drive layout ----
REPO_DIR   = '/content/Teleconnections-ViT'
DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
# ---------------------------------------------

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders
from models.losses    import TweedieLoss, MSELoss_cm


# ============================================================
#  GLOBAL TRAINING HYPERPARAMETERS
# ============================================================

SKIP_IF_EXISTS  = True
NUM_EPOCHS      = 20
BATCH_SIZE      = 8
MEMORY_YEARS    = 5
TWEEDIE_POWER   = 1.2
MAX_LR          = 3e-4
BASE_LR         = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
PATIENCE        = 20


# ============================================================
#  RUN MATRIX DEFINITIONS
# ============================================================

MODEL_CONFIGS = {
    'normal': {'embed_dim': 128, 'num_heads': 8, 'num_layers': 6,
               'd_ff': 512, 'dropout': 0.1},
    'small':  {'embed_dim': 128, 'num_heads': 4, 'num_layers': 3,
               'd_ff': 512, 'dropout': 0.2},
}

# Boolean flags only — in_channels and channel_mask_indices are derived
# automatically from the dataset after get_dataloaders() is called.
CHANNEL_CONFIGS = {
    'all':           {'use_spawners': True,  'use_recruits': True,  'use_temp': True},
    'sp_rec':        {'use_spawners': True,  'use_recruits': True,  'use_temp': False},
    'sp_temp':       {'use_spawners': True,  'use_recruits': False, 'use_temp': True},
    'rec_temp':      {'use_spawners': False, 'use_recruits': True,  'use_temp': True},
    'spawners_only': {'use_spawners': True,  'use_recruits': False, 'use_temp': False},
    'recruits_only': {'use_spawners': False, 'use_recruits': True,  'use_temp': False},
    'temp_only':     {'use_spawners': False, 'use_recruits': False, 'use_temp': True},
}

PREDICTION_MODES = {
    'normal':         {'incl_curr': True,  'lag': 0},
    'one_year_ahead': {'incl_curr': False, 'lag': 0},
    'lag5':           {'incl_curr': True,  'lag': 5},
}

CRITERIA = ['MSE', 'Tweedie']


# ============================================================
#  RUN MATRIX BUILDER
# ============================================================

def _needs_temp(ch):    return CHANNEL_CONFIGS[ch]['use_temp']
def _has_spawners(ch):  return CHANNEL_CONFIGS[ch]['use_spawners']
def _has_current_year_channel(ch):
    # one_year_ahead drops current-year channels (spawner + temp).
    # Only recruits_only has no current-year channel in either mode,
    # making one_year_ahead a no-op for it.
    ccfg = CHANNEL_CONFIGS[ch]
    return ccfg['use_spawners'] or ccfg['use_temp']


def build_run_matrix():
    runs = []

    # ---- Dummy: easy / medium / hard ----
    for level in ['easy', 'medium', 'hard']:
        for model_size in MODEL_CONFIGS:
            for ch_cfg in CHANNEL_CONFIGS:
                if _needs_temp(ch_cfg):
                    continue
                # Only run normal and one_year_ahead for dummy data
                for pred_mode in ['normal', 'one_year_ahead']:
                    if pred_mode == 'one_year_ahead' and not _has_current_year_channel(ch_cfg):
                        continue
                    for criterion in CRITERIA:
                        runs.append(dict(model_size=model_size, level=level,
                                         data_type='dummy', channel_cfg=ch_cfg,
                                         pred_mode=pred_mode, criterion=criterion))

    # ---- Real data: all 7 channel configs × all applicable modes ----
    for model_size in MODEL_CONFIGS:
        for ch_cfg in CHANNEL_CONFIGS:
            for pred_mode in PREDICTION_MODES:
                if pred_mode == 'one_year_ahead' and not _has_current_year_channel(ch_cfg):
                    continue
                for criterion in CRITERIA:
                    runs.append(dict(model_size=model_size, level='real',
                                     data_type='real', channel_cfg=ch_cfg,
                                     pred_mode=pred_mode, criterion=criterion))
    return runs


# ============================================================
#  HELPERS
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    # Dummy data (total 30 years). Subtract 5 from train for lag=5.
    return (18, 9, 3) if lag == 0


def get_run_dir(model_size, level, channel_cfg, pred_mode, criterion):
    return os.path.join(DRIVE_BASE, model_size, level, channel_cfg, pred_mode, criterion)


def compute_bias_correction(model, train_loader, device):
    """Full lognormal bias correction exp(μ + σ²/2) from training residuals."""
    model.eval()
    residuals = []
    with torch.no_grad():
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in train_loader:
            inputs        = inputs.to(device)
            targets       = targets.to(device)
            temporal_mask = temporal_mask.to(device)
            year_idx      = year_idx.to(device)
            spatial_mask  = spatial_mask.to(device)
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
            for i in range(targets.shape[0]):
                if valid_year[i] > 0:
                    vm  = spatial_mask[i] > 0
                    res = (targets[i, 0][vm] - outputs[i, 0][vm]).cpu().numpy()
                    residuals.append(res)
    residuals = np.concatenate(residuals)
    mu        = float(residuals.mean())
    sigma2    = float(residuals.var())
    return float(np.exp(mu + 0.5 * sigma2)), mu, sigma2


def save_training_curves(history, run_dir, title_str):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(epochs, history['train_loss'], color='steelblue', lw=2, label='Train')
    ax1.plot(epochs, history['val_loss'],   color='tomato',    lw=2, label='Val')
    ax1.set_title(title_str, fontsize=9); ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history['lr'], color='seagreen', lw=2)
    ax2.set_title('OneCycleLR'); ax2.set_xlabel('Epoch')
    ax2.set_ylabel('LR'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
#  SINGLE RUN
# ============================================================

def train_one_run(model_size, level, data_type, channel_cfg, pred_mode, criterion):
    run_dir         = get_run_dir(model_size, level, channel_cfg, pred_mode, criterion)
    checkpoint_path = os.path.join(run_dir, 'best_model.pt')

    if SKIP_IF_EXISTS and os.path.exists(checkpoint_path):
        print(f"  ⏭  Skipping: {os.path.relpath(run_dir, DRIVE_BASE)}")
        return True

    os.makedirs(run_dir, exist_ok=True)
    mcfg = MODEL_CONFIGS[model_size]
    ccfg = CHANNEL_CONFIGS[channel_cfg]
    pcfg = PREDICTION_MODES[pred_mode]
    t_years, v_years, te_years = get_year_splits(data_type, pcfg['lag'])

    title_str = f"{model_size} | {level} | {channel_cfg} | {pred_mode} | {criterion}"
    print(f"\n{'='*72}\n  {title_str}\n  → {run_dir}\n{'='*72}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    # -- Data --
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
        train_years=t_years, val_years=v_years, test_years=te_years,
        level=level, data_type=data_type,
        include_current_spawner=pcfg['incl_curr'],
        lag=pcfg['lag'],
        use_temp=ccfg['use_temp'],
        use_spawners=ccfg['use_spawners'],
        use_recruits=ccfg['use_recruits'],
    )

    # Read channel metadata directly from the dataset object
    ds                   = train_loader.dataset
    in_channels          = ds.in_channels
    channel_mask_indices = ds.channel_mask_indices
    print(f"  in_channels={in_channels}  mask_indices={channel_mask_indices}")

    # -- Model --
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=in_channels,
        embed_dim=mcfg['embed_dim'],
        num_heads=mcfg['num_heads'],
        num_layers=mcfg['num_layers'],
        d_ff=mcfg['d_ff'],
        dropout=mcfg['dropout'],
        channel_mask_indices=channel_mask_indices,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    # Decoder bias warm-start
    all_log_targets = []
    for _, targets, _, _, spatial_mask, valid_year in train_loader:
        for i in range(targets.shape[0]):
            if valid_year[i] > 0:
                all_log_targets.append(targets[i, 0][spatial_mask[i] > 0].numpy())
    mean_log = float(np.concatenate(all_log_targets).mean())
    nn.init.constant_(model.decoder.conv_out.bias, mean_log)
    print(f"  Decoder bias warm-started at {mean_log:.4f}")

    # -- Loss / optimiser / scheduler --
    criterion_fn = TweedieLoss(power=TWEEDIE_POWER) if criterion == 'Tweedie' else MSELoss_cm()
    optimizer    = torch.optim.AdamW(model.parameters(), lr=BASE_LR,
                                     weight_decay=WEIGHT_DECAY)
    scheduler    = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, epochs=NUM_EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.3,
        anneal_strategy='cos', div_factor=25, final_div_factor=1000,
    )

    # -- Training loop --
    best_val_loss    = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in train_loader:
            inputs, targets   = inputs.to(device),       targets.to(device)
            temporal_mask     = temporal_mask.to(device)
            year_idx          = year_idx.to(device)
            spatial_mask      = spatial_mask.to(device)
            valid_year        = valid_year.to(device)

            optimizer.zero_grad()
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)

            if criterion == 'Tweedie':
                loss = criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                    torch.expm1(targets).clamp(min=1e-6),
                                    mask=spatial_mask, sample_mask=valid_year)
            else:
                loss = criterion_fn(outputs, targets,
                                    mask=spatial_mask, sample_mask=valid_year)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in val_loader:
                inputs, targets   = inputs.to(device),       targets.to(device)
                temporal_mask     = temporal_mask.to(device)
                year_idx          = year_idx.to(device)
                spatial_mask      = spatial_mask.to(device)
                valid_year        = valid_year.to(device)
                outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
                if criterion == 'Tweedie':
                    val_loss += criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                             torch.expm1(targets).clamp(min=1e-6),
                                             mask=spatial_mask,
                                             sample_mask=valid_year).item()
                else:
                    val_loss += criterion_fn(outputs, targets,
                                             mask=spatial_mask,
                                             sample_mask=valid_year).item()

        avg_val    = val_loss / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['lr'].append(current_lr)

        improved = ''
        if avg_val < best_val_loss:
            best_val_loss    = avg_val
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            improved = '  ✓ saved'
        else:
            patience_counter += 1

        print(f"  Ep {epoch+1:3d}/{NUM_EPOCHS}  train={avg_train:.4f}  "
              f"val={avg_val:.4f}  lr={current_lr:.2e}  "
              f"pat={patience_counter}/{PATIENCE}{improved}")

        if patience_counter >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    # -- Bias correction (MSE only) --
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    history.update({'bias_correction': 1.0, 'bias_mu': 0.0, 'bias_sigma2': 0.0})
    if criterion == 'MSE':
        bc, mu, sigma2 = compute_bias_correction(model, train_loader, device)
        history.update({'bias_correction': bc, 'bias_mu': mu, 'bias_sigma2': sigma2})
        print(f"  Bias correction: {bc:.4f}  (μ={mu:.4f}, σ²={sigma2:.4f})")

    # -- Test loss --
    test_loss = 0.0
    with torch.no_grad():
        for inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year in test_loader:
            inputs, targets   = inputs.to(device),       targets.to(device)
            temporal_mask     = temporal_mask.to(device)
            year_idx          = year_idx.to(device)
            spatial_mask      = spatial_mask.to(device)
            valid_year        = valid_year.to(device)
            outputs = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask)
            if criterion == 'Tweedie':
                test_loss += criterion_fn(torch.expm1(outputs).clamp(min=1e-6),
                                          torch.expm1(targets).clamp(min=1e-6),
                                          mask=spatial_mask,
                                          sample_mask=valid_year).item()
            else:
                test_loss += criterion_fn(outputs, targets,
                                          mask=spatial_mask,
                                          sample_mask=valid_year).item()
    history['test_loss'] = test_loss / len(test_loader)
    print(f"  Test loss: {history['test_loss']:.4f}")

    # Save channel metadata in JSON for eval script to recover without re-loading data
    history['channel_cfg_meta'] = {
        'in_channels':          in_channels,
        'channel_mask_indices': channel_mask_indices,
        'use_spawners':         ccfg['use_spawners'],
        'use_recruits':         ccfg['use_recruits'],
        'use_temp':             ccfg['use_temp'],
        'incl_curr':            pcfg['incl_curr'],
        'lag':                  int(pcfg['lag']),
        'embed_dim':            mcfg['embed_dim'],
        'num_heads':            mcfg['num_heads'],
        'num_layers':           mcfg['num_layers'],
        'd_ff':                 mcfg['d_ff'],
    }

    with open(os.path.join(run_dir, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    save_training_curves(history, run_dir, title_str)
    print(f"  ✅  Done.  Best val loss: {best_val_loss:.4f}")
    return True


# ============================================================
#  ENTRY POINT
# ============================================================

if __name__ == '__main__':
    runs = build_run_matrix()
    n    = len(runs)

    print(f"\n{'='*72}")
    print(f"  CrabTransformer — Master Training Run")
    print(f"  Total planned runs: {n}  |  SKIP_IF_EXISTS={SKIP_IF_EXISTS}")
    print(f"  Output root: {DRIVE_BASE}")
    print(f"{'='*72}")
    for i, r in enumerate(runs, 1):
        print(f"  [{i:3d}/{n}]  {r['model_size']:6s}  {r['level']:6s}  "
              f"{r['channel_cfg']:14s}  {r['pred_mode']:15s}  {r['criterion']}")

    print(f"\nStarting …\n")
    failed = []
    for i, run in enumerate(runs, 1):
        print(f"\n[RUN {i}/{n}]")
        try:
            train_one_run(**run)
        except Exception as e:
            print(f"  ❌  {e}")
            traceback.print_exc()
            failed.append({'run': i, 'config': run, 'error': str(e)})

    print(f"\n{'='*72}")
    print(f"Training complete.  {n - len(failed)}/{n} runs succeeded.")
    if failed:
        print("Failed runs:")
        for f in failed:
            print(f"  [{f['run']}]  {f['config']}  →  {f['error']}")
    print(f"{'='*72}\n")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from models.model import CrabTransformer
from data.data_helper import get_dataloaders

# Ensure you import your custom losses!
# from utils.losses import TweedieLoss, MSELoss_cm

def test_overfit_on_real_data(data_type='real', level='real', loss_criterion='MSE', lag=0):
    print("=" * 70)
    print(f"OVERFITTING TEST ({data_type}/{level}, {loss_criterion}, LAG={lag})")
    print("=" * 70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ---> FIX: Dynamically set splits based on lag <---
    if data_type == 'real':
        if lag == 0:
            t_years, v_years, te_years = 24, 8, 4
        elif lag == 5:
            t_years, v_years, te_years = 21, 6, 4
        else:
            raise ValueError("Only lag 0 and 5 are supported.")
    else:
        t_years, v_years, te_years = 18, 9, 3

    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=8, memory_years=5,
        train_years=t_years, val_years=v_years, test_years=te_years,
        data_type=data_type, lag=lag
    )

    model = CrabTransformer(
        grid_size=50, patch_size=5, in_channels=17,
        embed_dim=128, num_heads=8, num_layers=6, d_ff=512, dropout=0
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")

    all_targets = []
    for inputs, tgts, temp_mask, y_idx, spat_mask, val_year in train_loader:
        for i in range(tgts.shape[0]):
            if val_year[i] > 0: # Skip 2020
                valid = tgts[i, 0][spat_mask[i] > 0]
                all_targets.append(valid.cpu().numpy())

    mean_log_target = np.concatenate(all_targets).mean()
    nn.init.constant_(model.decoder.conv_out.bias, mean_log_target)
    print(f"Decoder bias set to {mean_log_target:.3f}")

    # =======================================================
    # Grab ONE sample that is GUARANTEED to not be 2020
    # =======================================================
    for inputs_b, targets_b, temp_mask_b, y_idx_b, spat_mask_b, val_year_b in train_loader:
        valid_indices = torch.where(val_year_b > 0)[0]

        if len(valid_indices) > 0:
            idx = valid_indices[0]
            inputs = inputs_b[idx:idx+1].to(device)
            targets = targets_b[idx:idx+1].to(device)
            temporal_mask = temp_mask_b[idx:idx+1].to(device)
            year_idx = y_idx_b[idx:idx+1].to(device)
            spatial_mask = spat_mask_b[idx:idx+1].to(device)
            valid_year = val_year_b[idx:idx+1].to(device)
            break

    print(f"Selected Year Index {year_idx.item()} for overfitting test.")

    # ---> FIX: Ensure criterion is actually defined <---
    if loss_criterion == 'Tweedie':
        # Uncomment and use your custom Tweedie loss here
        # criterion = TweedieLoss(power=1.5)
        pass
        targets = torch.expm1(targets).clamp(min=1e-6)
    elif loss_criterion == 'MSE':
        # criterion = MSELoss_cm()
        criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    print("\n🚀 Training for 1000 epochs on a single sample...")
    model.train()

    for epoch in range(1000):
        optimizer.zero_grad()

        output, _ = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask, return_attention=True)

        if loss_criterion == 'Tweedie':
            output_loss = torch.expm1(output).clamp(min=1e-6)
            loss = criterion(output_loss, targets) # add mask=spatial_mask if your custom loss supports it
        elif loss_criterion == 'MSE':
            loss = criterion(output, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"   Epoch {epoch+1:4d} | Loss: {loss.item():.8f}")

    # Final verification
    model.eval()
    with+ torch.no_grad():
        final_output, _ = model(inputs, year_idx, temporal_mask, spatial_mask=spatial_mask, return_attention=True)

        valid_mask = spatial_mask.unsqueeze(1) > 0

        if loss_criterion == 'Tweedie':
            pred_np = torch.expm1(final_output)[valid_mask].cpu().numpy().flatten()
        else:
            pred_np = final_output[valid_mask].cpu().numpy().flatten()

        targ_np = targets[valid_mask].cpu().numpy().flatten()
        corr = np.corrcoef(pred_np, targ_np)[0, 1]

    print("\n" + "=" * 70)
    print("🏁 FINAL DIAGNOSTIC VERDICT:")
    print("=" * 70)

    if corr > 0.98:
        print(f"✅ SUCCESS! Final Correlation: {corr:.4f}")
    else:
        print(f"❌ FAILURE. Final Correlation: {corr:.4f}")

if __name__ == "__main__":
    test_overfit_on_real_data(data_type='real', loss_criterion='MSE', lag=0)
    test_overfit_on_real_data(data_type='real', loss_criterion='MSE', lag=5)

OVERFITTING TEST (real/real, MSE, LAG=0)
Loaded 100 bootstrap samples, 24 years each
Applying Log-Scaling to Spawners/Recruits ONLY.
Loaded 100 bootstrap samples, 8 years each
  Using 5 years of historical data from previous split
Applying Log-Scaling to Spawners/Recruits ONLY.
Loaded 100 bootstrap samples, 4 years each
  Using 5 years of historical data from previous split
Applying Log-Scaling to Spawners/Recruits ONLY.
Sinusoidal temporal encoding initialized:
  Pre-computed years: 0-49
  Can dynamically compute beyond year 50 ✓
Initializing model weights...
✓ Applying Bias Initialization Surgery to Decoder (-1.0)
✓ Weight initialization complete
Trainable parameters: 1,321,249
Decoder bias set to 2.819
Selected Year Index 10 for overfitting test.

🚀 Training for 1000 epochs on a single sample...
   Epoch  100 | Loss: 0.11154789
   Epoch  200 | Loss: 0.02331997
   Epoch  300 | Loss: 0.00999909
   Epoch  400 | Loss: 0.00742243
   Epoch  500 | Loss: 0.00535685
   Epoch  600 | Loss: 0.0

In [ ]:
"""
run_batch_evaluation.py  —  Evaluate all trained CrabTransformer checkpoints.

Auto-discovers checkpoints by scanning for best_model.pt under DRIVE_BASE.
Parses run config entirely from the directory path (no manual config list).

For every sample, metrics are computed TWICE:
  raw_*    — back-transformed predictions, no additional thresholding
  thresh_* — both pred and target zeroed below ZERO_THRESHOLD (14.23)
             before all metric calculations

Outputs (saved to SAVE_DIR)
----------------------------
  full_results.csv     one row per (model_size/level/channel_cfg/pred_mode/
                       criterion/phase/year/bootstrap)
  summary_results.csv  mean ± SD per (config × phase)
  plots/{run_id}/
    abundance.png      median pred vs obs trajectory with bootstrap IQR
    spatial_grids.png  representative spatial maps for train / val / test
"""

import os
import sys
import json
import glob
import traceback

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# ---- adjust to your layout ----
REPO_DIR   = '/content/Teleconnections-ViT'
DRIVE_BASE = '/content/drive/MyDrive/Teleconnection_ViT/model_outputs'
SAVE_DIR   = '/content/drive/MyDrive/Teleconnection_ViT/analysis'
# --------------------------------

sys.path.insert(0, REPO_DIR)
from models.model     import CrabTransformer
from data.data_helper import get_dataloaders

ZERO_THRESHOLD  = 14.23
TOP_K_FRACTION  = 0.10
BATCH_SIZE      = 8
MEMORY_YEARS    = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================================================
#  YEAR SPLITS (must match train.py)
# ============================================================

def get_year_splits(data_type, lag):
    if data_type == 'real':
        return (24, 8, 4) if lag == 0 else (21, 6, 4)
    # Dummy data (total 30 years). Subtract 5 from train for lag=5.
    return (18, 9, 3) if lag == 0


# ============================================================
#  METRICS
# ============================================================

def compute_metrics(p_flat: np.ndarray, t_flat: np.ndarray,
                    train_mean: float = None) -> dict:
    """
    Full metric suite for one spatial sample (1-D arrays of valid cells).

    Returns a flat dict.  Returns an empty dict for degenerate inputs.
    """
    n = len(p_flat)
    if n == 0:
        return {}

    # Spatial correlation
    if np.std(p_flat) > 0 and np.std(t_flat) > 0:
        spear, _ = spearmanr(p_flat, t_flat)
        pear,  _ = pearsonr(p_flat,  t_flat)
    else:
        spear = pear = 0.0

    # Error decomposition
    diff          = p_flat - t_flat
    mae           = float(np.mean(np.abs(diff)))
    rmse          = float(np.sqrt(np.mean(diff ** 2)))
    bias          = float(np.mean(diff))
    unbiased_rmse = float(np.sqrt(max(rmse ** 2 - bias ** 2, 0.0)))
    bias_frac_pct = float(bias ** 2 / (rmse ** 2 + 1e-10) * 100)

    # Abundance
    pred_total        = float(p_flat.sum())
    obs_total         = float(t_flat.sum())
    abundance_capture = pred_total / (obs_total + 1e-6)
    pct_abund_error   = (pred_total - obs_total) / (obs_total + 1e-6) * 100

    # Top-10 % Jaccard
    k            = max(1, int(n * TOP_K_FRACTION))
    pred_top     = set(np.argsort(p_flat)[-k:])
    obs_top      = set(np.argsort(t_flat)[-k:])
    intersect    = len(pred_top & obs_top)
    union        = len(pred_top | obs_top)
    jaccard      = intersect / union if union > 0 else 0.0

    # Zero-cell classification (threshold = 14.23 applied BEFORE calling this
    # function for the thresh variant; the raw variant has no prior zeroing)
    pred_zero    = p_flat < ZERO_THRESHOLD
    obs_zero     = t_flat < ZERO_THRESHOLD
    tp = float(np.sum( pred_zero &  obs_zero))
    fp = float(np.sum( pred_zero & ~obs_zero))
    fn = float(np.sum(~pred_zero &  obs_zero))
    precision    = tp / (tp + fp + 1e-10)
    recall       = tp / (tp + fn + 1e-10)
    f1           = 2 * precision * recall / (precision + recall + 1e-10)

    # Skill scores vs climatological mean baseline
    if train_mean is not None:
        base_mse  = float(np.mean((t_flat - train_mean) ** 2))
        base_mae  = float(np.mean(np.abs(t_flat - train_mean)))
        skill_mse = float(1.0 - np.mean(diff ** 2) / (base_mse + 1e-10))
        skill_mae = float(1.0 - mae               / (base_mae  + 1e-10))
    else:
        skill_mse = skill_mae = float('nan')

    return {
        'spearman':          float(spear),
        'pearson':           float(pear),
        'mae':               mae,
        'rmse':              rmse,
        'bias':              bias,
        'unbiased_rmse':     unbiased_rmse,
        'bias_frac_pct':     bias_frac_pct,
        'pred_total':        pred_total,
        'obs_total':         obs_total,
        'abundance_capture': abundance_capture,
        'pct_abund_error':   float(pct_abund_error),
        'top10_jaccard':     float(jaccard),
        'zero_precision':    float(precision),
        'zero_recall':       float(recall),
        'zero_f1':           float(f1),
        'skill_mse':         skill_mse,
        'skill_mae':         skill_mae,
    }


# ============================================================
#  PLOTS
# ============================================================

def save_abundance_plot(records, run_id, plot_dir, train_end, val_end, n_years):
    df  = pd.DataFrame(records)
    ann = df.groupby('year').agg(
        pred_median=('raw_pred_total', 'median'),
        pred_p25   =('raw_pred_total', lambda x: np.percentile(x, 25)),
        pred_p75   =('raw_pred_total', lambda x: np.percentile(x, 75)),
        obs_median =('raw_obs_total',  'median'),
    ).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(ann['year'], ann['pred_p25'], ann['pred_p75'],
                    alpha=0.25, color='steelblue', label='Pred IQR')
    ax.plot(ann['year'], ann['pred_median'], 'b-o', ms=4, lw=2, label='Pred median')
    ax.plot(ann['year'], ann['obs_median'],  'k-o', ms=4, lw=2, label='Obs median')
    ax.axvline(train_end - 0.5, color='grey', ls=':', lw=1.5)
    ax.axvline(val_end   - 0.5, color='grey', ls=':', lw=1.5)
    t = ax.get_xaxis_transform()
    ax.text(train_end / 2,               0.97, 'TRAIN', transform=t,
            ha='center', color='seagreen',   fontsize=9, fontweight='bold')
    ax.text((train_end + val_end) / 2,   0.97, 'VAL',   transform=t,
            ha='center', color='darkorange', fontsize=9, fontweight='bold')
    ax.text((val_end + n_years) / 2,     0.97, 'TEST',  transform=t,
            ha='center', color='crimson',    fontsize=9, fontweight='bold')
    ax.set_xlabel('Year index'); ax.set_ylabel('Total abundance')
    ax.set_title(f'Abundance — {run_id}', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'abundance.png'), dpi=150, bbox_inches='tight')
    plt.close()


def save_spatial_grid(records, run_id, plot_dir, train_end, val_end):
    phase_target = {'TRAIN': train_end // 2,
                    'VAL':   train_end + 2,
                    'TEST':  val_end   + 1}
    phase_color  = {'TRAIN': 'seagreen', 'VAL': 'darkorange', 'TEST': 'crimson'}

    fig, axes = plt.subplots(3, 3, figsize=(11, 11))
    titles    = ['Spawner / ch-0 (log)', 'True Recruits (log)', 'Pred Recruits (log)']

    for row, (phase, tgt_yr) in enumerate(phase_target.items()):
        cands = sorted([r for r in records if r['phase'] == phase],
                       key=lambda r: abs(r['year'] - tgt_yr))
        if not cands:
            for col in range(3):
                axes[row, col].axis('off')
            continue
        r   = cands[0]
        col_0_img = r.get('input_ch0_log', np.zeros((50, 50)))
        imgs  = [col_0_img, r['target_log'], r['pred_log']]
        for col, (img, ttl) in enumerate(zip(imgs, titles)):
            ax = axes[row, col]
            ax.imshow(img, cmap='viridis' if col == 0 else 'plasma',
                      vmin=0, vmax=8, interpolation='nearest')
            if row == 0:
                ax.set_title(ttl, fontsize=9)
            ax.set_ylabel(f'Yr {r["year"]} [{phase}]' if col == 0 else '')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor(phase_color[phase])
                sp.set_linewidth(2.5); sp.set_visible(True)

    plt.suptitle(f'Spatial grids — {run_id}', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'spatial_grids.png'), dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
#  EVALUATE ONE RUN
# ============================================================

def evaluate_run(run_dir: str) -> list:
    """
    Load checkpoint, run inference on all three splits, return per-sample records.
    Config is recovered from the directory path + training_history.json.
    """
    # -- Parse path ---
    rel   = os.path.relpath(run_dir, DRIVE_BASE)
    parts = rel.split(os.sep)
    if len(parts) != 5:
        raise ValueError(f"Unexpected depth: {rel}  (need 5 levels)")
    model_size, level, channel_cfg, pred_mode, criterion = parts
    data_type = 'real' if level == 'real' else 'dummy'
    run_id    = rel.replace(os.sep, '/')

    # -- Load training history (contains channel metadata) ---
    history_path = os.path.join(run_dir, 'training_history.json')
    if not os.path.exists(history_path):
        raise FileNotFoundError(f"No training_history.json in {run_dir}")
    with open(history_path) as f:
        hist = json.load(f)

    bias_correction = hist.get('bias_correction', 1.0)
    meta            = hist.get('channel_cfg_meta', {})

    # Recover channel metadata from JSON (saved by train.py)
    in_channels          = meta['in_channels']
    channel_mask_indices = meta['channel_mask_indices']
    use_spawners         = meta['use_spawners']
    use_recruits         = meta['use_recruits']
    use_temp             = meta['use_temp']
    incl_curr            = meta['incl_curr']
    lag                  = meta['lag']
    embed_dim            = meta.get('embed_dim', 128)
    num_heads            = meta.get('num_heads', 8)
    num_layers           = meta.get('num_layers', 6)
    d_ff                 = meta.get('d_ff', 512)

    t_years, v_years, te_years = get_year_splits(data_type, lag)

    # -- Model ---
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=in_channels,
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        d_ff=d_ff,
        dropout=0.0,
        channel_mask_indices=channel_mask_indices,
    ).to(device)
    model.load_state_dict(
        torch.load(os.path.join(run_dir, 'best_model.pt'), map_location=device)
    )
    model.eval()

    # -- Data ---
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=BATCH_SIZE, memory_years=MEMORY_YEARS,
        train_years=t_years, val_years=v_years, test_years=te_years,
        level=level, data_type=data_type,
        include_current_spawner=incl_curr,
        lag=lag,
        use_temp=use_temp,
        use_spawners=use_spawners,
        use_recruits=use_recruits,
    )

    # Spatial validity mask
    if data_type == 'real':
        valid_mask = np.load('data/real/output/spatial_mask.npy') > 0
    else:
        valid_mask = np.ones((50, 50), dtype=bool)

    # Training mean for skill scores
    train_vals = []
    with torch.no_grad():
        for _, targets, _, _, spatial_mask, valid_year in train_loader:
            for i in range(targets.shape[0]):
                if valid_year[i] > 0:
                    train_vals.append(np.expm1(targets[i, 0].numpy())[valid_mask])
    train_mean = float(np.concatenate(train_vals).mean()) if train_vals else None

    n_years_total = t_years + v_years + te_years
    loaders = [
        ('TRAIN', train_loader),
        ('VAL',   val_loader),
        ('TEST',  test_loader),
    ]

    records = []
    seen_years_in_plot = {}          # phase → set of years already stored for plots

    with torch.no_grad():
        for phase, loader in loaders:
            seen_years_in_plot[phase] = set()

            for batch in loader:
                inputs, targets, temporal_mask, year_idx, spatial_mask, valid_year = batch

                inputs        = inputs.to(device)
                temporal_mask = temporal_mask.to(device)
                year_idx_dev  = year_idx.to(device)
                spatial_mask  = spatial_mask.to(device)

                outputs = model(inputs, year_idx_dev, temporal_mask,
                                spatial_mask=spatial_mask)

                p_log = outputs.cpu().numpy()
                t_log = targets.numpy()
                y_abs = year_idx.numpy()
                vy    = valid_year.numpy()

                for i in range(p_log.shape[0]):
                    if vy[i] == 0:
                        continue

                    yr = int(y_abs[i])

                    # Back-transform: exp(log1p_pred) × bias_correction - 1
                    p_raw = np.clip(np.exp(p_log[i, 0]) * bias_correction - 1.0,
                                    0, None)
                    t_raw = np.expm1(t_log[i, 0])
                    p_raw[~valid_mask] = 0.0
                    t_raw[~valid_mask] = 0.0

                    p_flat = p_raw[valid_mask]
                    t_flat = t_raw[valid_mask]

                    # ---- RAW metrics ----
                    raw_m = compute_metrics(p_flat, t_flat, train_mean)

                    # ---- THRESHOLDED metrics ----
                    p_thr = p_flat.copy();  p_thr[p_thr < ZERO_THRESHOLD] = 0.0
                    t_thr = t_flat.copy();  t_thr[t_thr < ZERO_THRESHOLD] = 0.0
                    thr_m = compute_metrics(p_thr, t_thr, train_mean)

                    rec = {
                        'model_size':  model_size,
                        'level':       level,
                        'channel_cfg': channel_cfg,
                        'pred_mode':   pred_mode,
                        'criterion':   criterion,
                        'phase':       phase,
                        'year':        yr,
                        'bias_correction': bias_correction,
                        'raw_pred_total': float(p_flat.sum()),
                        'raw_obs_total':  float(t_flat.sum()),
                    }
                    for k, v in raw_m.items():
                        rec[f'raw_{k}'] = v
                    for k, v in thr_m.items():
                        rec[f'thresh_{k}'] = v

                    # Store grids for the first bootstrap per year (for plots)
                    if yr not in seen_years_in_plot[phase]:
                        seen_years_in_plot[phase].add(yr)
                        # channel 0 is first input channel if spawners present,
                        # otherwise recruits t-1; just show input[:,0] as proxy
                        rec['input_ch0_log'] = inputs[i, 0].cpu().numpy()
                        rec['target_log']    = t_log[i, 0]
                        rec['pred_log']      = p_log[i, 0]

                    records.append(rec)

    # -- Per-run plots --
    plot_dir = os.path.join(SAVE_DIR, 'plots', run_id.replace('/', '_'))
    os.makedirs(plot_dir, exist_ok=True)
    try:
        # Filter to records that have grid arrays
        plot_recs = [r for r in records if 'target_log' in r]
        save_abundance_plot(records, run_id, plot_dir,
                            t_years, t_years + v_years, n_years_total)
        save_spatial_grid(plot_recs, run_id, plot_dir,
                          t_years, t_years + v_years)
    except Exception as e:
        print(f"  ⚠  Plot failed: {e}")

    # Drop large array fields before returning (keep CSVs lean)
    for r in records:
        r.pop('input_ch0_log', None)
        r.pop('target_log',    None)
        r.pop('pred_log',      None)

    return records


# ============================================================
#  SUMMARY
# ============================================================

def make_summary(df: pd.DataFrame) -> pd.DataFrame:
    group_cols  = ['model_size', 'level', 'channel_cfg', 'pred_mode', 'criterion', 'phase']
    metric_cols = [c for c in df.columns if c.startswith('raw_') or c.startswith('thresh_')]
    means  = df.groupby(group_cols)[metric_cols].mean().add_suffix('_mean')
    stds   = df.groupby(group_cols)[metric_cols].std().add_suffix('_std')
    counts = df.groupby(group_cols)[metric_cols[0]].count().rename('n_samples')
    return pd.concat([means, stds, counts], axis=1).reset_index().round(4)


def print_overview(df: pd.DataFrame):
    test = df[df['phase'] == 'TEST']
    if test.empty:
        print("No TEST records."); return
    agg = (test.groupby(['model_size', 'level', 'channel_cfg', 'pred_mode', 'criterion'])
               .agg(spearman=('raw_spearman',         'mean'),
                    spear_sd =('raw_spearman',         'std'),
                    abund    =('raw_abundance_capture','mean'),
                    skill    =('raw_skill_mse',        'mean'),
                    n        =('raw_spearman',         'count'))
               .reset_index()
               .sort_values('spearman', ascending=False))
    pd.set_option('display.float_format', '{:.3f}'.format)
    pd.set_option('display.max_rows', 300)
    print("\n=== TEST SET OVERVIEW (raw Spearman, sorted desc) ===")
    print(agg.to_string(index=False))
    pd.reset_option('display.float_format')


# ============================================================
#  MAIN
# ============================================================

def run_all_evaluations():
    os.makedirs(SAVE_DIR, exist_ok=True)

    ckpt_paths = sorted(glob.glob(
        os.path.join(DRIVE_BASE, '*', '*', '*', '*', '*', 'best_model.pt')
    ))
    run_dirs = [os.path.dirname(p) for p in ckpt_paths]

    print(f"\n{'='*72}")
    print(f"  CrabTransformer — Batch Evaluation")
    print(f"  Found {len(run_dirs)} trained models under {DRIVE_BASE}")
    print(f"  Saving to {SAVE_DIR}")
    print(f"{'='*72}\n")

    all_records = []
    failed      = []

    for i, run_dir in enumerate(run_dirs, 1):
        rel = os.path.relpath(run_dir, DRIVE_BASE)
        print(f"[{i:3d}/{len(run_dirs)}]  {rel}")
        try:
            recs = evaluate_run(run_dir)
            all_records.extend(recs)
            print(f"  ✅  {len(recs)} samples")
        except Exception as e:
            print(f"  ❌  {e}")
            traceback.print_exc()
            failed.append({'index': i, 'dir': rel, 'error': str(e)})

    if not all_records:
        print("No records collected — nothing to save.")
        return

    df = pd.DataFrame(all_records)

    full_path = os.path.join(SAVE_DIR, 'full_results.csv')
    df.to_csv(full_path, index=False)
    print(f"\n✅  Full results  → {full_path}  ({len(df):,} rows)")

    summary = make_summary(df)
    summ_path = os.path.join(SAVE_DIR, 'summary_results.csv')
    summary.to_csv(summ_path, index=False)
    print(f"✅  Summary       → {summ_path}  ({len(summary):,} rows)")

    print_overview(df)

    if failed:
        print(f"\n⚠  {len(failed)} runs failed:")
        for f in failed:
            print(f"  [{f['index']}] {f['dir']}  →  {f['error']}")

    print(f"\n{'='*72}")
    print(f"Evaluation complete. "
          f"{len(run_dirs) - len(failed)}/{len(run_dirs)} runs succeeded.")
    print(f"{'='*72}\n")


if __name__ == '__main__':
    run_all_evaluations()

In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')  # Fix Agg
import matplotlib.pyplot as plt
import torch
import numpy as np
import os
import json
from data.data_helper import get_dataloaders

DRIVE_DIR = "/content/drive/MyDrive/Teleconnection_ViT"
SAVE_DIR = os.path.join(DRIVE_DIR, "analysis/batch_evaluation")
os.makedirs(SAVE_DIR, exist_ok=True)

def generate_comparison_panel(levels=['easy', 'medium', 'hard'], num_reps=100,
                              data_type='dummy'):
    n_panels = len(levels)
    fig, axes = plt.subplots(1, n_panels, figsize=(8 * n_panels, 8), sharey=True)
    if n_panels == 1:
        axes = [axes]

    TRAIN_YEARS = 22 if data_type == 'real' else 22
    VAL_YEARS = 9
    TEST_YEARS = 3
    N_YEARS = TRAIN_YEARS + VAL_YEARS + TEST_YEARS
    TRAIN_END = TRAIN_YEARS
    VAL_END = TRAIN_YEARS + VAL_YEARS

    # Load spatial mask for real data
    if data_type == 'real':
        valid_mask = np.load("data/real/output/spatial_mask.npy") > 0  # [50, 50]
    else:
        valid_mask = np.ones((50, 50), dtype=bool)

    def add_phase_regions(ax):
        ax.axvline(TRAIN_END - 0.5, color="grey", ls=":", lw=1.2, alpha=0.8)
        ax.axvline(VAL_END - 0.5, color="grey", ls=":", lw=1.2, alpha=0.8)
        ax.axvspan(-0.5, TRAIN_END - 0.5, color='green', alpha=0.03)
        ax.axvspan(TRAIN_END - 0.5, VAL_END - 0.5, color='orange', alpha=0.03)
        ax.axvspan(VAL_END - 0.5, N_YEARS - 0.5, color='red', alpha=0.03)

    for col, level in enumerate(levels):
        eval_level = 'real' if data_type == 'real' else level

        train_loader, val_loader, test_loader = get_dataloaders(
            batch_size=1, level=eval_level, transform='log',
            train_years=TRAIN_YEARS, val_years=VAL_YEARS, test_years=TEST_YEARS,
            data_type=data_type
        )

        full_truth, full_spawners = [], []

        with torch.no_grad():
            for loader in [train_loader, val_loader, test_loader]:
                for batch in loader:
                    inputs = batch[0]
                    targets = batch[1]

                    # Sum only over valid ocean cells
                    spawner_raw = torch.expm1(inputs[:, 0]).cpu().numpy()
                    recruit_raw = torch.expm1(targets[:, 0]).cpu().numpy()

                    full_spawners.append(
                        spawner_raw[:, valid_mask].sum(axis=1).flatten()
                    )
                    full_truth.append(
                        recruit_raw[:, valid_mask].sum(axis=1).flatten()
                    )

        def process_reps(data_list):
            arr = np.concatenate(data_list).reshape(num_reps, N_YEARS)
            return (
                np.percentile(arr, 50, axis=0),
                np.percentile(arr, 5, axis=0),
                np.percentile(arr, 95, axis=0),
            )

        mid_truth, lo_truth, hi_truth = process_reps(full_truth)
        mid_spwn,  lo_spwn,  hi_spwn  = process_reps(full_spawners)

        time_x = np.arange(N_YEARS)
        ax = axes[col]
        add_phase_regions(ax)

        ax.plot(time_x, mid_spwn, color='green', label='Spawners (median)',
                lw=2.5, ls='--')
        ax.fill_between(time_x, lo_spwn, hi_spwn, color='green', alpha=0.08)

        ax.plot(time_x, mid_truth, color='black', label='Recruits (median)',
                lw=2, zorder=10)
        ax.fill_between(time_x, lo_truth, hi_truth, color='black', alpha=0.08)

        ax.set_title(f"SCENARIO: {eval_level.upper()}", fontweight='bold')
        ax.set_xlabel("Year")
        if col == 0:
            ax.set_ylabel("Total Abundance")
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"data_comparison_panel_{data_type}.png"))
    plt.show()

# Dummy data
# generate_comparison_panel(levels=['easy', 'medium', 'hard'], data_type='dummy')

# Real data
generate_comparison_panel(levels=['real'], num_reps=100, data_type='real')

Plot for Maia

In [ ]:
"""
=============================================================================
Integrated Gradients for CrabTransformer (Targeted Runs)
=============================================================================
Post-hoc attribution analysis. Run AFTER training is complete.
"""

import os
import json
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

# Adjust based on your environment
REPO_DIR   = '/content/Teleconnections-ViT'
import sys
sys.path.insert(0, REPO_DIR)

from models.model import CrabTransformer
from data.data_helper import get_dataloaders

# =============================================================================
# CONFIGURATION
# =============================================================================

DRIVE_DIR  = '/content/drive/MyDrive/Teleconnection_ViT'
OUTPUTS_DIR = os.path.join(DRIVE_DIR, 'model_outputs')
SAVE_DIR   = os.path.join(DRIVE_DIR, 'analysis/attribution')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHANNEL_NAMES = [
    'Spawner (t)', 'Spawner (t-1)', 'Spawner (t-2)', 'Spawner (t-3)', 'Spawner (t-4)', 'Spawner (t-5)',
    'Recruit (t-1)', 'Recruit (t-2)', 'Recruit (t-3)', 'Recruit (t-4)', 'Recruit (t-5)',
    'Temp (t)', 'Temp (t-1)', 'Temp (t-2)', 'Temp (t-3)', 'Temp (t-4)', 'Temp (t-5)'
]

os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# STEP 1: COMPUTE BASELINE (mean field from training data)
# =============================================================================

def compute_baseline(train_loader):
    print('  Computing baseline (mean training field)...')
    all_inputs = []
    for inputs, _, _, _, _, _ in train_loader:
        all_inputs.append(inputs)

    stacked = torch.cat(all_inputs, dim=0)
    baseline = stacked.mean(dim=0, keepdim=True)

    print(f'    Baseline from {stacked.shape[0]} samples')
    print(f'    Range: [{baseline.min():.3f}, {baseline.max():.3f}]')
    return baseline


# =============================================================================
# STEP 2: ORGANIZE SAMPLES BY YEAR
# =============================================================================

def collect_samples_by_year(loader):
    by_year = {}

    for inputs, targets, mask, year_idx, spat_mask, val_year in loader:
        batch_size = inputs.shape[0]
        for i in range(batch_size):
            if val_year[i] == 0:
                continue

            yr = int(year_idx[i].item())
            if yr not in by_year:
                by_year[yr] = []
            by_year[yr].append((
                inputs[i:i+1],
                targets[i:i+1],
                mask[i:i+1],
                year_idx[i:i+1],
            ))

    for yr in sorted(by_year.keys()):
        print(f'    Year {yr}: {len(by_year[yr])} bootstrap samples')

    return by_year


# =============================================================================
# STEP 3: INTEGRATED GRADIENTS (core computation)
# =============================================================================

def integrated_gradients(model, actual_input, baseline, year_idx,
                         temporal_mask, n_steps=50, output_fn=None):
    if output_fn is None:
        output_fn = lambda out: out.mean()

    delta = actual_input - baseline
    accumulated_grads = torch.zeros_like(actual_input)

    for step in range(n_steps):
        alpha = step / n_steps
        interpolated = baseline + alpha * delta
        interpolated = interpolated.detach().clone().requires_grad_(True)

        output = model(interpolated, year_idx, temporal_mask)
        scalar_output = output_fn(output)

        model.zero_grad()
        scalar_output.backward()

        accumulated_grads += interpolated.grad.detach()

    attribution = (accumulated_grads / n_steps) * delta

    return attribution.squeeze(0).cpu().numpy()


# =============================================================================
# STEP 4: RUN IG PER YEAR, AVERAGE ACROSS BOOTSTRAPS
# =============================================================================

def compute_yearly_attributions(model, by_year, baseline, n_steps=50, output_fn=None):
    baseline = baseline.to(DEVICE)
    yearly_attr = {}
    yearly_inputs = {}

    for yr in sorted(by_year.keys()):
        samples = by_year[yr]
        n_bootstraps = len(samples)
        print(f'\n  Year {yr}: running IG on {n_bootstraps} bootstraps...')

        attr_list = []
        input_list = []

        for idx, (inp, tgt, msk, yr_idx) in enumerate(samples):
            inp_dev  = inp.to(DEVICE)
            yr_dev   = yr_idx.to(DEVICE)
            msk_dev  = msk.to(DEVICE)

            attr = integrated_gradients(
                model, inp_dev, baseline,
                yr_dev, msk_dev,
                n_steps=n_steps, output_fn=output_fn
            )
            attr_list.append(attr)
            input_list.append(inp.squeeze(0).numpy())

            if (idx + 1) % 25 == 0:
                print(f'    {idx + 1}/{n_bootstraps} done')

        yearly_attr[yr]   = np.stack(attr_list).mean(axis=0)
        yearly_inputs[yr] = np.stack(input_list).mean(axis=0)

        print(f'    Year {yr} done. Attr range: [{yearly_attr[yr].min():.6f}, {yearly_attr[yr].max():.6f}]')

    return yearly_attr, yearly_inputs


# =============================================================================
# STEP 5: VISUALIZATION
# =============================================================================

def plot_year_panel(yr, mean_input, mean_attr, meta, out_name):
    use_temp = meta['use_temp']
    n_channels = meta['in_channels']

    rows_per_half = 3 if use_temp else 2
    total_rows = rows_per_half * 2

    fig, axes = plt.subplots(total_rows, 6, figsize=(24, 4 * total_rows))

    abs_attr = np.abs(mean_attr)
    vmax_attr = np.percentile(abs_attr, 99)

    # ── Top Half: Mean Input Channels ──
    for c in range(n_channels):
        row, col = c // 6, c % 6
        ax = axes[row, col]
        ax.imshow(mean_input[c], cmap='viridis', vmin=0, vmax=8 if c < 11 else None)
        ax.set_title(f'Input: {CHANNEL_NAMES[c]}', fontsize=8, fontweight='bold')
        ax.axis('off')

    # Aggregate input spot
    ax = axes[n_channels // 6, n_channels % 6]
    input_agg = np.abs(mean_input).sum(axis=0)
    ax.imshow(input_agg, cmap='viridis')
    ax.set_title('Input: Aggregate', fontsize=8, fontweight='bold')
    ax.axis('off')

    # ── Bottom Half: Attribution Channels ──
    for c in range(n_channels):
        row, col = rows_per_half + (c // 6), c % 6
        ax = axes[row, col]
        im = ax.imshow(mean_attr[c], cmap='RdBu_r', vmin=-vmax_attr, vmax=vmax_attr)
        ax.set_title(f'Attr: {CHANNEL_NAMES[c]}', fontsize=8, fontweight='bold')
        ax.axis('off')

    # Aggregate |attribution| spot
    ax = axes[rows_per_half + (n_channels // 6), n_channels % 6]
    agg_attr = abs_attr.sum(axis=0)
    im_agg = ax.imshow(agg_attr, cmap='hot')
    ax.set_title('Attr: Aggregate |IG|', fontsize=8, fontweight='bold')
    ax.axis('off')

    fig.suptitle(
        f'Year {yr} — {out_name}\n'
        f'Top: Mean Input   |   Bottom: Attribution (red=+recruit, blue=-recruit)',
        fontsize=16, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    path = os.path.join(SAVE_DIR, f'ig_year{yr}_{out_name}.png')
    plt.savefig(path, dpi=150)
    print(f'  Saved: {path}')
    plt.close()


def plot_grand_average(yearly_attr, yearly_inputs, meta, out_name):
    years = sorted(yearly_attr.keys())
    grand_attr  = np.stack([yearly_attr[yr] for yr in years]).mean(axis=0)
    grand_input = np.stack([yearly_inputs[yr] for yr in years]).mean(axis=0)

    plot_year_panel('AVG', grand_input, grand_attr, meta, out_name)


def plot_channel_bar_yearly(yearly_attr, meta, out_name):
    years = sorted(yearly_attr.keys())

    data = {}
    for yr in years:
        totals = np.abs(yearly_attr[yr]).sum(axis=(1, 2))
        data[f'Yr {yr}'] = totals / totals.sum() * 100

    grand = np.stack([yearly_attr[yr] for yr in years]).mean(axis=0)
    grand_totals = np.abs(grand).sum(axis=(1, 2))
    data['Average'] = grand_totals / grand_totals.sum() * 100

    # Read all flags
    use_temp = meta['use_temp']
    use_recruits = meta['use_recruits']
    incl_curr = meta['incl_curr']
    n_channels = meta['in_channels']

    # --- THE FIX: Conditionally build the entire active names list ---
    active_names = []

    # 1. Spawner names (always present in these 3 models)
    if incl_curr: active_names.append('Spawner (t)')
    active_names.extend(['Spawner (t-1)', 'Spawner (t-2)', 'Spawner (t-3)', 'Spawner (t-4)', 'Spawner (t-5)'])

    # 2. Recruit names
    if use_recruits:
        active_names.extend(['Recruit (t-1)', 'Recruit (t-2)', 'Recruit (t-3)', 'Recruit (t-4)', 'Recruit (t-5)'])

    # 3. Temp names
    if use_temp:
        if incl_curr: active_names.append('Temp (t)')
        active_names.extend(['Temp (t-1)', 'Temp (t-2)', 'Temp (t-3)', 'Temp (t-4)', 'Temp (t-5)'])
    # ----------------------------------------------------------------

    x = np.arange(n_channels)
    n_groups = len(data)
    width = 0.8 / n_groups
    colors = plt.cm.tab10(np.linspace(0, 1, n_groups))

    fig, ax = plt.subplots(figsize=(16, 6))

    for i, (label, pcts) in enumerate(data.items()):
        offset = (i - n_groups / 2 + 0.5) * width
        bars = ax.bar(x + offset, pcts, width, label=label,
                      color=colors[i], alpha=0.8, edgecolor='black',
                      linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(active_names, rotation=45, ha='right')
    ax.set_ylabel('Share of Total Attribution (%)')
    ax.set_title(
        f'Channel Attribution by Year: {out_name}',
        fontsize=14, fontweight='bold'
    )
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3, ls='--')
    plt.tight_layout()

    path = os.path.join(SAVE_DIR, f'ig_bar_yearly_{out_name}.png')
    plt.savefig(path, dpi=150)
    print(f'  Saved: {path}')
    plt.close()


# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

def run_targeted_ig(model_size, level, channel_cfg, pred_mode, criterion, phase='TEST', n_steps=50):

    out_name = f"{model_size}_{level}_{channel_cfg}_{pred_mode}_{criterion}"

    print(f'\n{"="*70}')
    print(f' Running IG for: {out_name}')
    print(f' Phase: {phase} | Steps: {n_steps}')
    print(f'{"="*70}')

    # 1. Resolve exact directory matching train.py output structure
    ckpt_dir = os.path.join(OUTPUTS_DIR, model_size, level, channel_cfg, pred_mode, criterion)
    ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')
    history_path = os.path.join(ckpt_dir, 'training_history.json')

    if not os.path.exists(ckpt_path) or not os.path.exists(history_path):
        print(f'  ⚠️ Missing model or history file in: {ckpt_dir}\n  Skipping...')
        return

    # 2. Load architectural metadata dynamically saved during training!
    with open(history_path, 'r') as f:
        meta = json.load(f)['channel_cfg_meta']

    print(f"  Loaded Meta -> Channels: {meta['in_channels']}, Lag: {meta['lag']}, Temp: {meta['use_temp']}")

    # 3. Load Model
    model = CrabTransformer(
        grid_size=50, patch_size=5,
        in_channels=meta['in_channels'],
        embed_dim=meta['embed_dim'],
        num_heads=meta['num_heads'],
        num_layers=meta['num_layers'],
        d_ff=meta['d_ff'],
        dropout=0,
        channel_mask_indices=meta['channel_mask_indices'] # Required for CrabTransformer!
    ).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    # 4. Load Data (Always loads real data structure per these targeted runs)
    t_years, v_years, te_years = (24, 8, 4) if meta['lag'] == 0 else (21, 6, 4)

    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=8, memory_years=5,
        train_years=t_years, val_years=v_years, test_years=te_years,
        data_type='real', level=level,
        include_current_spawner=meta['incl_curr'],
        lag=meta['lag'],
        use_temp=meta['use_temp'],
        use_spawners=meta['use_spawners'],
        use_recruits=meta['use_recruits']
    )

    baseline = compute_baseline(train_loader)

    if phase == 'TEST':
        target_loader = test_loader
    elif phase == 'VAL':
        target_loader = val_loader
    else:
        target_loader = train_loader

    print(f'\n  Collecting {phase} samples by year...')
    by_year = collect_samples_by_year(target_loader)

    yearly_attr, yearly_inputs = compute_yearly_attributions(
        model, by_year, baseline, n_steps=n_steps
    )

    # ── Save raw arrays ──
    for yr in sorted(yearly_attr.keys()):
        np.save(os.path.join(SAVE_DIR, f'ig_attr_yr{yr}_{out_name}.npy'), yearly_attr[yr])
        np.save(os.path.join(SAVE_DIR, f'ig_input_yr{yr}_{out_name}.npy'), yearly_inputs[yr])

    print(f'  Raw arrays saved to {SAVE_DIR}')

    print(f'\n  Generating visualizations...')
    for yr in sorted(yearly_attr.keys()):
        plot_year_panel(yr, yearly_inputs[yr], yearly_attr[yr], meta, out_name)

    plot_grand_average(yearly_attr, yearly_inputs, meta, out_name)
    plot_channel_bar_yearly(yearly_attr, meta, out_name)

if __name__ == '__main__':
    # =========================================================================
    # TARGETED RUNS: Only the 3 configurations needed for the paper!
    # =========================================================================

    TARGET_MODELS = [
        # 1. The "Nowcast" Baseline
        {'model_size': 'normal', 'level': 'real', 'channel_cfg': 'all', 'pred_mode': 'normal', 'criterion': 'MSE'},

        # 2. The Operational Forecast (1-Year-Ahead)
        {'model_size': 'small', 'level': 'real', 'channel_cfg': 'all', 'pred_mode': 'one_year_ahead', 'criterion': 'MSE'},

        # 3. The Biological Lag Proof (5-Year Lag)
        {'model_size': 'normal', 'level': 'real', 'channel_cfg': 'sp_temp', 'pred_mode': 'lag5', 'criterion': 'MSE'},

        #4. Temperature only:
        {'model_size': 'small', 'level': 'real', 'channel_cfg': 'temp_only', 'pred_mode': 'normal', 'criterion': 'MSE'}
    ]

    for run_params in TARGET_MODELS:
        run_targeted_ig(**run_params, phase='TEST', n_steps=50)